[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-customerseg.ipynb)

# Full Project: Credit Card Customer Segmentation

*AIBits Academy · Machine Learning End To End · Full Project*

8,950 credit card holders, 18 usage-behaviour features, and a genuine K-Means model-selection exercise — silhouette score picks a smaller k than the elbow curve alone would suggest.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['customer_segmentation.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the credit-card usage data** (8,950 customers, 18 columns).

In [ ]:
df = pd.read_csv('customer_segmentation.csv')
print(df.shape)
df.head()

> **Business Problem**
>
> A bank wants to design different marketing offers and credit-limit policies for different types of cardholders, instead of treating its entire customer base identically — but has no pre-existing labels for "type of customer," so this has to be discovered from usage behaviour alone.

> **Dataset**
>
> **8,950 customers, 18 features** covering balance, purchase types (one-off vs. installment), cash advances, credit limit, payments, and tenure. [Dataset source →](https://statso.io/2022/11/23/customer-segmentation-case-study/)

## Step 1 — Scaling Is Non-Negotiable Here

`BALANCE` ranges into the thousands while `BALANCE_FREQUENCY` is a 0–1 ratio — without scaling, K-Means (which is entirely distance-based) would let balance dominate every cluster assignment. Two missing-value columns are median-imputed first:

In [ ]:
from sklearn.preprocessing import StandardScaler

df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median())
df['CREDIT_LIMIT'] = df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())

X = df.drop(columns=['CUST_ID'])
X_scaled = StandardScaler().fit_transform(X)

## Step 2 — Choosing k: Elbow vs. Silhouette Disagree

The elbow curve (inertia) decreases smoothly all the way to k=8 with no sharp "elbow" — a common real-world outcome that makes the elbow method alone unreliable here. The silhouette score tells a clearer story:

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    print(f"k={k}  inertia={km.inertia_:.1f}  silhouette={silhouette_score(X_scaled, labels):.4f}")

Inertia keeps dropping at every k (as it mathematically must), but silhouette peaks clearly at **k=3** (0.2510) before dropping at k=4–5 and only partially recovering by k=8. Inertia alone would tempt picking a larger k just because the curve is still falling; silhouette — which explicitly measures how well-separated the clusters are, not just how tight — says 3 genuinely distinct groups is the better-supported choice.

## Visualizing Why the Two Signals Disagree

Left: inertia falls smoothly with no elbow. Right: silhouette score clearly peaks at k=3 (gold) before dipping and only partially recovering — the more trustworthy signal for a business-actionable segment count.

## Step 3 — Three Clearly Interpretable Segments

| Cluster | Size | Avg. Balance | Avg. Purchases | Avg. Cash Advance | Avg. Credit Limit | Profile |
|---|---|---|---|---|---|---|
| 0 | 1,275 | \$2,182 | \$4,187 | \$450 | \$7,643 | **Big Spenders** — high purchases, rarely uses cash advance |
| 1 | 6,114 | \$808 | \$496 | \$339 | \$3,267 | **Low-Engagement** — the majority (68%), low activity across the board |
| 2 | 1,561 | \$4,024 | \$389 | \$3,917 | \$6,729 | **Cash-Advance Reliant** — high balance carried mainly via cash advances, not purchases |

Cluster 2 is the actionable surprise: high balance and high credit limit, but low purchase activity — these customers are using their card almost entirely for cash advances (typically the highest-fee, highest-interest way to use a credit card), not everyday spending. That's a fundamentally different — and higher-risk — usage pattern than Cluster 0's high-purchase "Big Spenders," despite both carrying substantial balances.

> **💡 Business Application**
>
> Cluster 1 (68% of customers, low engagement across every metric) is the natural target for an activation campaign — a low starting balance means there's real room to grow this segment's usage. Cluster 2's cash-advance-reliant pattern is a candidate for a financial-wellness outreach rather than a spending incentive, since encouraging more cash advances would increase, not reduce, this segment's risk exposure.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Which columns had gaps?

Re-read the file into `raw` and store in `missing` a dict of the columns that have missing values and how many (`{column: count}`).

In [ ]:
raw = pd.read_csv("customer_segmentation.csv")
missing = {}   # TODO


In [ ]:
try:
    check("two columns", set(missing) == {"MINIMUM_PAYMENTS", "CREDIT_LIMIT"})
    check("counts", missing["MINIMUM_PAYMENTS"] == 313 and missing["CREDIT_LIMIT"] == 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
raw = pd.read_csv("customer_segmentation.csv")
n = raw.isna().sum()
missing = n[n > 0].to_dict()

```

</details>

### Exercise 2 · Medium · Cluster sizes for k = 3

Fit `KMeans(n_clusters=3, random_state=42, n_init=10)` on `X_scaled`. Store the labels in `labels3` and the sorted cluster sizes in `sizes3`.

In [ ]:
labels3 = sizes3 = None   # TODO


In [ ]:
try:
    check("every customer assigned", len(labels3) == 8950 and sum(sizes3) == 8950)
    check("three clusters", len(sizes3) == 3)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
labels3 = km3.labels_
sizes3 = sorted(np.bincount(labels3).tolist())

```

</details>

### Exercise 3 · Stretch · Profile the big spenders

Add the cluster labels to `df` as `cluster`, compute the mean of every numeric column per cluster into `profile`, and store the id of the cluster with the highest mean `PURCHASES` in `spender_cluster`.

In [ ]:
profile = spender_cluster = None   # TODO


In [ ]:
try:
    check("one row per cluster", profile.shape[0] == 3)
    check("consistent", spender_cluster == profile["PURCHASES"].idxmax())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
df["cluster"] = labels3
profile = df.drop(columns=["CUST_ID"]).groupby("cluster").mean()
spender_cluster = int(profile["PURCHASES"].idxmax())

```

A cluster is only useful once you can describe it in business terms - the profile table is that description.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Credit Card Customer Segmentation**.*